# Gold Layer: `fact_cash_balances` -- Derived Fact Table

| Property | Value |
|:---------|:------|
| **Target Table** | `gold.fact_cash_balances` |
| **Expected Rows** | 1,088,273 |
| **Grain** | One row per (AccountID, Date) -- daily running balance |
| **MERGE Key** | `(SK_AccountID, SK_DateID)` |
| **Derived From** | `gold.fact_cash_transactions` |

### Schema
| Column | Type | Description |
|:-------|:-----|:------------|
| `AccountID` | BIGINT | Natural account key |
| `DateValue` | DATE | Balance date |
| `SK_CustomerID` | BIGINT | FK to `dim_customer` (via dim_account temporal join) |
| `SK_AccountID` | BIGINT NOT NULL | FK to `dim_account` (temporal join, part of MERGE key) |
| `SK_DateID` | BIGINT NOT NULL | FK to `dim_date` (part of MERGE key) |
| `Cash` | DECIMAL(15,2) | Running cash balance (cumulative sum) |
| `_batch` | STRING | Batch identifier |

### Algorithm
1. Read `fact_cash_transactions` (AccountID = SK_AccountID which is CT_CA_ID natural key)
2. Extract DATE from TransactionDatetime -> DateValue
3. Aggregate daily: SUM(Amount) per (AccountID, DateValue)
4. Compute running cumulative SUM per account ordered by date -> Cash
5. Temporal join to `dim_account`: AccountID = accountid AND DateValue >= effectivedate AND DateValue < enddate
6. Join to `dim_date` on DateValue -> SK_DateID
7. Select final schema + MERGE into gold

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import (
    col, sum as _sum, count, max as _max, min as _min, lit,
    to_date, current_timestamp
)
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType, LongType
from delta.tables import DeltaTable

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONFIGURATION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CATALOG       = "charles_schwab_retailbrokerage_dev_team_lemma"
GOLD_SCHEMA   = "gold"

SOURCE_TABLE  = f"{CATALOG}.{GOLD_SCHEMA}.fact_cash_transactions"
TARGET_TABLE  = f"{CATALOG}.{GOLD_SCHEMA}.fact_cash_balances"
DIM_ACCOUNT   = f"{CATALOG}.{GOLD_SCHEMA}.dim_account"
DIM_DATE      = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"
EXPECTED_ROWS = 1088273

spark.sql(f"USE CATALOG {CATALOG}")

print("\n" + "="*70)
print("  GOLD.FACT_CASH_BALANCES -- Pipeline Configuration")
print("="*70)
print(f"  Catalog       : {CATALOG}")
print(f"  Source Table  : {SOURCE_TABLE}")
print(f"  Target Table  : {TARGET_TABLE}")
print(f"  Dim Account   : {DIM_ACCOUNT}")
print(f"  Dim Date      : {DIM_DATE}")
print(f"  Expected Rows : {EXPECTED_ROWS:,}")
print(f"  MERGE Key     : (SK_AccountID, SK_DateID)")
print(f"  Grain         : One row per (AccountID, Date)")
print("="*70)

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 1: Read Source -- gold.fact_cash_transactions
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Source schema (from fact_cash_transactions):
#    SK_AccountID        BIGINT     -> This is actually CT_CA_ID (natural key)
#    TransactionDatetime TIMESTAMP  -> Extract DATE for daily aggregation
#    Amount              DECIMAL(12,2) -> SUM per day, then cumulative
#    Description         STRING     -> Not needed for balances
#    _batch              STRING     -> Carried forward
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 1: Read Source -- gold.fact_cash_transactions")
print("="*70)

txn_df = spark.table(SOURCE_TABLE)
source_count = txn_df.count()

print(f"\n  Source rows: {source_count:,}")
print(f"  Batch distribution:")
txn_df.groupBy("_batch").count().orderBy("_batch").show()

print("  Sample source data (first 5):")
display(txn_df.orderBy("SK_AccountID", "TransactionDatetime").limit(5))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 2: Daily Aggregation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Aggregate transactions by (AccountID, Date):
#    - SUM(Amount) = net daily cash flow
#    - MAX(_batch) = batch attribution for that day
#  Note: SK_AccountID in fact_cash_transactions = CT_CA_ID (natural key)
#        We rename it to AccountID for clarity.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 2: Daily Aggregation")
print("="*70)

# Extract date and rename SK_AccountID -> AccountID (natural key)
daily_agg_df = (
    txn_df
    .withColumn("DateValue", to_date(col("TransactionDatetime")))
    .groupBy(
        col("SK_AccountID").alias("AccountID"),
        "DateValue"
    )
    .agg(
        _sum("Amount").cast(DecimalType(15, 2)).alias("DailyAmount"),
        _max("_batch").alias("_batch")
    )
)

daily_count = daily_agg_df.count()
print(f"\n  Daily aggregated rows: {daily_count:,}")
print(f"  (This is the expected final row count = {EXPECTED_ROWS:,})")
status = "PASS" if daily_count == EXPECTED_ROWS else "CHECK"
print(f"  Match: {status}")

print("\n  Sample daily aggregation (first 10):")
display(daily_agg_df.orderBy("AccountID", "DateValue").limit(10))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 3: Running Cumulative Balance
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  For each account, compute cumulative sum of daily amounts
#  ordered by date. This gives the running cash balance.
#
#  Window: PARTITION BY AccountID ORDER BY DateValue
#          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 3: Running Cumulative Balance")
print("="*70)

# Window: cumulative sum per account ordered by date
balance_window = (
    Window.partitionBy("AccountID")
    .orderBy("DateValue")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

balance_df = daily_agg_df.withColumn(
    "Cash",
    _sum("DailyAmount").over(balance_window).cast(DecimalType(15, 2))
).drop("DailyAmount")

print(f"\n  Running balance computed.")
print(f"  Rows: {balance_df.count():,}")

print("\n  Sample running balances for first account (first 10 dates):")
display(
    balance_df
    .orderBy("AccountID", "DateValue")
    .limit(10)
)

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 4: Dimension Joins (dim_account + dim_date)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Temporal join to dim_account:
#    AccountID = dim_account.accountid
#    DateValue >= dim_account.effectivedate
#    DateValue < dim_account.enddate
#  -> Retrieves SK_AccountID and SK_CustomerID for correct SCD-2 version
#
#  Direct join to dim_date:
#    DateValue = dim_date.DateValue
#  -> Retrieves SK_DateID
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 4: Dimension Joins")
print("="*70)

# Read dimensions
dim_account_df = spark.table(DIM_ACCOUNT).select(
    col("accountid"),
    col("SK_AccountID").alias("da_SK_AccountID"),
    col("SK_CustomerID"),
    col("effectivedate"),
    col("enddate")
)

dim_date_df = spark.table(DIM_DATE).select(
    col("SK_DateID"),
    col("DateValue").alias("dd_DateValue")
)

print(f"\n  dim_account rows: {dim_account_df.count():,}")
print(f"  dim_date rows: {dim_date_df.count():,}")

# Temporal join to dim_account
print("\n  Joining to dim_account (temporal)...")
with_account_df = balance_df.join(
    dim_account_df,
    (balance_df["AccountID"] == dim_account_df["accountid"]) &
    (balance_df["DateValue"] >= dim_account_df["effectivedate"]) &
    (balance_df["DateValue"] < dim_account_df["enddate"]),
    "left"
).select(
    balance_df["AccountID"],
    balance_df["DateValue"],
    col("SK_CustomerID"),
    col("da_SK_AccountID").alias("SK_AccountID"),
    balance_df["Cash"],
    balance_df["_batch"]
)

print(f"  After dim_account join: {with_account_df.count():,} rows")

# Join to dim_date
print("  Joining to dim_date...")
final_df = with_account_df.join(
    dim_date_df,
    with_account_df["DateValue"] == dim_date_df["dd_DateValue"],
    "left"
).select(
    col("AccountID"),
    with_account_df["DateValue"],
    col("SK_CustomerID"),
    col("SK_AccountID").cast(LongType()),
    col("SK_DateID").cast(LongType()),
    col("Cash"),
    col("_batch")
)

final_count = final_df.count()
status = "PASS" if final_count == EXPECTED_ROWS else "MISMATCH"
print(f"\n  +{'─'*54}+")
print(f"  |  Final row count : {final_count:>10,}                   |")
print(f"  |  Expected        : {EXPECTED_ROWS:>10,}                   |")
print(f"  |  Status          : {status:>10}                   |")
print(f"  +{'─'*54}+")

# Check for nulls in required fields
sk_acct_nulls = final_df.filter(col("SK_AccountID").isNull()).count()
sk_date_nulls = final_df.filter(col("SK_DateID").isNull()).count()
print(f"\n  SK_AccountID nulls: {sk_acct_nulls} (from unmatched temporal join)")
print(f"  SK_DateID nulls: {sk_date_nulls} (from unmatched date join)")

print("\n  Sample with dimension keys (first 10):")
display(final_df.orderBy("AccountID", "DateValue").limit(10))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 5: Write to Gold (MERGE for idempotency)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  MERGE ON: (SK_AccountID, SK_DateID)
#  First run: CREATE via overwrite
#  Subsequent runs: MERGE for idempotent reprocessing
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 5: Write to Gold (Idempotent)")
print("="*70)

if spark.catalog.tableExists(TARGET_TABLE):
    print(f"\n  Mode: MERGE (table exists)")
    print(f"  Key : (SK_AccountID, SK_DateID)")
    
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)
    delta_target.alias("tgt").merge(
        final_df.alias("src"),
        """tgt.SK_AccountID = src.SK_AccountID 
           AND tgt.SK_DateID = src.SK_DateID"""
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    
    print("  MERGE complete.")
else:
    print(f"\n  Mode: CREATE (first run)")
    print(f"  Table: {TARGET_TABLE}")
    
    final_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TARGET_TABLE)
    
    print("  CREATE complete.")

target_count = spark.table(TARGET_TABLE).count()
status = "PASS" if target_count == EXPECTED_ROWS else "FAIL"
print(f"\n  Target count: {target_count:,} (expected: {EXPECTED_ROWS:,}) [{status}]")

print("\n  Delta Table Version History:")
display(spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 5"))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 6: Data Quality Validation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 6: Data Quality Validation")
print("="*70)

gold_df = spark.table(TARGET_TABLE)
results = []

# DQ-1: Row Count
row_count = gold_df.count()
p = row_count == EXPECTED_ROWS
results.append(("DQ-1", "Row Count", f"{row_count:,} vs {EXPECTED_ROWS:,}", "PASS" if p else "FAIL"))

# DQ-2: SK_AccountID Not Null
sk_acct_nulls = gold_df.filter(col("SK_AccountID").isNull()).count()
results.append(("DQ-2", "SK_AccountID Not Null", f"{sk_acct_nulls} nulls", "PASS" if sk_acct_nulls == 0 else "WARN"))

# DQ-3: SK_DateID Not Null
sk_date_nulls = gold_df.filter(col("SK_DateID").isNull()).count()
results.append(("DQ-3", "SK_DateID Not Null", f"{sk_date_nulls} nulls", "PASS" if sk_date_nulls == 0 else "WARN"))

# DQ-4: Composite Key Uniqueness (SK_AccountID, SK_DateID)
dup_count = (
    gold_df.groupBy("SK_AccountID", "SK_DateID")
    .count()
    .filter(col("count") > 1)
    .count()
)
results.append(("DQ-4", "MERGE Key Unique", f"{dup_count} duplicates", "PASS" if dup_count == 0 else "FAIL"))

# DQ-5: Cash Not Null
cash_nulls = gold_df.filter(col("Cash").isNull()).count()
results.append(("DQ-5", "Cash Not Null", f"{cash_nulls} nulls", "PASS" if cash_nulls == 0 else "FAIL"))

# DQ-6: AccountID Not Null
acct_nulls = gold_df.filter(col("AccountID").isNull()).count()
results.append(("DQ-6", "AccountID Not Null", f"{acct_nulls} nulls", "PASS" if acct_nulls == 0 else "FAIL"))

# DQ-7: Cash range
cash_stats = gold_df.select(
    _min("Cash").alias("min_cash"),
    _max("Cash").alias("max_cash")
).collect()[0]
results.append(("DQ-7", "Cash Range", f"min={cash_stats['min_cash']}, max={cash_stats['max_cash']}", "INFO"))

# DQ-8: Date range
date_stats = gold_df.select(
    _min("DateValue").alias("min_dt"),
    _max("DateValue").alias("max_dt")
).collect()[0]
results.append(("DQ-8", "Date Range", f"{date_stats['min_dt']} to {date_stats['max_dt']}", "INFO"))

# DQ-9: SK_CustomerID nulls (informational - some accounts may not match)
sk_cust_nulls = gold_df.filter(col("SK_CustomerID").isNull()).count()
results.append(("DQ-9", "SK_CustomerID Nulls", f"{sk_cust_nulls} nulls", "INFO"))

# Print results
print("\n  {:<6} {:<25} {:<40} {}".format("Check", "Description", "Result", "Status"))
print("  " + "-"*85)
for check_id, desc, result, status in results:
    print(f"  {check_id:<6} {desc:<25} {result:<40} {status}")

print("\n  Table Detail:")
display(spark.sql(f"DESCRIBE DETAIL {TARGET_TABLE}"))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 7: Operations Logging
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 7: Operations Logging")
print("="*70)

try:
    # Get run_id from upstream source
    run_id = spark.table(f"{CATALOG}.silver.cash_transactions").select("_run_id").first()[0]
    
    log_pipeline_recon(
        spark=spark,
        run_id=run_id,
        batch_id="ALL",
        domain="ACCOUNT",
        table_name="fact_cash_balances",
        source_layer="gold",
        target_layer="gold",
        source_count=source_count,
        target_count=target_count
    )
    
    log_audit_event(
        spark=spark,
        run_id=run_id,
        batch="ALL",
        layer="gold",
        table_name="fact_cash_balances",
        operation="MERGE" if spark.catalog.tableExists(TARGET_TABLE) else "CREATE",
        rows_affected=target_count
    )
    
    print(f"\n  log_pipeline_recon: source={source_count:,} -> target={target_count:,}")
    print(f"  log_audit_event: rows={target_count:,}")
except Exception as e:
    print(f"  [WARN] Operations logging failed: {e}")
    print(f"  (Non-blocking -- gold table was written successfully)")

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  VERIFICATION: Final State Summary
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  VERIFICATION: gold.fact_cash_balances Final State")
print("="*70)

verify_df = spark.table(TARGET_TABLE)

print(f"\n  Total rows: {verify_df.count():,} (expected: {EXPECTED_ROWS:,})")

print("\n  Batch Distribution:")
verify_df.groupBy("_batch").count().orderBy("_batch").show()

print("  Balance Statistics:")
display(
    verify_df.select(
        count("*").alias("total_rows"),
        _min("Cash").alias("min_balance"),
        _max("Cash").alias("max_balance"),
        count("SK_CustomerID").alias("with_customer"),
        count("SK_AccountID").alias("with_account"),
        count("SK_DateID").alias("with_date")
    )
)

print("  Unique accounts:", verify_df.select("AccountID").distinct().count())

print("\n  Top 10 accounts by balance date count:")
display(
    verify_df.groupBy("AccountID")
    .agg(
        count("*").alias("date_count"),
        _min("Cash").alias("min_balance"),
        _max("Cash").alias("max_balance")
    )
    .orderBy(col("date_count").desc())
    .limit(10)
)

print("\n  Sample balances (first 15):")
display(verify_df.orderBy("AccountID", "DateValue").limit(15))